# **DATA PREPROCESSING TABULAR**

---

## **1. Import Libraries**

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error

from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from itertools import combinations

## **2. Load Raw Data For Preprocessing**

Dựa trên những phát hiện quan trọng thu được ở bước khám phá dữ liệu (EDA) — giai đoạn tiền xử lý được triển khai nhằm chuẩn hóa cấu trúc dữ liệu, đảm bảo mỗi biến phản ánh đúng bản chất định tính hoặc định lượng của nó. 

Khác với các bộ dữ liệu đã được làm sạch sẵn, dữ liệu điều tra dân số thực tế thường chứa các ký tự dị thường (`?`). Bước khởi tạo này sẽ nạp dữ liệu và ép kiểu các ký tự lỗi về chuẩn `np.nan` của thư viện Pandas.

In [2]:
adult_path = "../data/raw/adult.csv"
df_adult = pd.read_csv(adult_path)
df_adult = df_adult.replace(regex=r'^\s*\?\s*$', value=np.nan)

num_cols = ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']
for col in num_cols:
    if col in df_adult.columns:
        df_adult[col] = pd.to_numeric(df_adult[col], errors='coerce')

Dữ liệu được đọc từ tệp `adult.csv` thành công và được khởi tạo dưới dạng cấu trúc dữ liệu bảng (tabular), lưu trữ trong biến DataFrame `df_adult`.

## **3. Controlled Missing Value Imputation**

**Mục tiêu:** Thay vì áp dụng mù quáng một thuật toán điền khuyết, nhóm tiến hành đánh giá định lượng hiệu quả của 5 chiến lược thông qua môi trường mô phỏng (simulation). Cụ thể, nhóm sẽ giả lập mất mát dữ liệu (MCAR) trên một biến số học hoàn chỉnh, sau đó dùng các thuật toán để nội suy và tính toán sai số RMSE so với Ground Truth.

### **3.1 Theoretical Foundation & Mathematical Formulas**

Trước khi thực thi, việc thấu hiểu nền tảng toán học và giả định của từng thuật toán là bắt buộc để đánh giá rủi ro sai lệch (bias) mà chúng có thể mang lại.

#### 1. Arithmetic Mean Imputation 

* **Lý thuyết:** Đây là kỹ thuật điền khuyết đơn giản nhất dành cho dữ liệu liên tục. Nó sử dụng thước đo xu hướng tập trung, giả định rằng giá trị trung bình là giá trị có khả năng xuất hiện cao nhất đối với một quan sát lấy ngẫu nhiên từ phân phối chuẩn (Gaussian). Tuy nhiên, nó cực kỳ nhạy cảm với ngoại lai, làm giảm nhân tạo phương sai của biến và thu hẹp khoảng tin cậy.
* **Toán học:** Giá trị trung bình mẫu $\bar{x}$ được tính bằng:
  $$\bar{x} = \frac{\sum_{i=1}^{n} x_i}{n}$$

#### 2. Median Imputation

* **Lý thuyết:** Một giải pháp thay thế mạnh mẽ cho phương pháp trung bình, đặc biệt khi dữ liệu có độ lệch (skewness) cao hoặc chứa ngoại lai ngoại lai (như biến `capital.gain` trong tập dữ liệu này). Tuy nhiên, nó vẫn làm biến dạng phân phối do tạo ra một spike nhân tạo tại vùng trung tâm.
* **Toán học:** Dữ liệu quan sát được sắp xếp tăng dần: $x_{(1)}, x_{(2)}, \dots, x_{(n)}$.
  * Nếu $n$ lẻ: Trung vị là $x_{(\frac{n+1}{2})}$.
  * Nếu $n$ chẵn: Trung vị là $\frac{x_{(\frac{n}{2})} + x_{(\frac{n}{2} + 1)}}{2}$.

#### 3. Mode Imputation

* **Lý thuyết:** Giá trị xuất hiện với tần suất cao nhất. Chủ yếu dùng cho biến phân loại (Categorical). Mặc dù đơn giản, phương pháp này thất bại nếu phân phối đa đỉnh (multimodal) và bỏ qua hoàn toàn mối tương quan giữa các đặc trưng.
* **Toán học:** Là giá trị $x_i$ cực đại hóa hàm tần suất $f(x)$.

#### 4. k-Nearest Neighbors (k-NN) Imputation

* **Lý thuyết:** k-NN là thuật toán lazy learning phi tham số. Thay vì dùng một hằng số toàn cục, nó nội suy bằng cách tìm $k$ quan sát hoàn chỉnh giống nhất (hàng xóm) trong không gian đặc trưng.
  * $k=3$: Bám sát mẫu cục bộ, rủi ro quá khớp (overfitting) với nhiễu.
  * $k=5$: Trạng thái cân bằng tối ưu, lọc được nhiễu nhưng vẫn giữ được cấu trúc.
  * $k=10$: Rủi ro trơn hóa quá mức (underfitting), dần hội tụ về trung bình toàn cục.
* **Toán học:** Tính khoảng cách (ví dụ: Euclidean) giữa các điểm dữ liệu:
  $$d(a, b) = \sqrt{\sum_{i=1}^{m} (a_i - b_i)^2}$$

#### 5. Multiple Imputation by Chained Equations (MICE)

* **Lý thuyết:** Còn gọi là Fully Conditional Specification (FCS). Hoạt động dưới giả định MAR, MICE lập mô hình từng biến bị khuyết như một hàm của các biến còn lại thông qua một chuỗi các phương trình hồi quy. MICE bảo toàn hình dáng phân phối và tính đa đỉnh (multimodality) xuất sắc.
* **Toán học:** Mô hình hồi quy cơ bản cho biến liên tục:
  $$x_{j, obs} = \beta_0 + \mathbf{X}_{-j, obs} \boldsymbol{\beta} + \epsilon_j, \quad \epsilon_j \sim N(0, \sigma^2)$$
  *Luật kết hợp Rubin (Rubin's Rules):* Tổng hợp từ $H$ bộ dữ liệu với phương sai tổng cộng $T$:
  $$T = \bar{V}_H + \left(1 + \frac{1}{H}\right) B_H$$

### **3.2 MCAR Injection**

Nhóm lựa chọn thuộc tính `age` (Tuổi) làm đối tượng thử nghiệm vì cột này nguyên bản không có giá trị khuyết (hoàn hảo để làm Ground Truth) và có tương quan với nhiều biến kinh tế khác. Nhóm sẽ chủ động xóa ngẫu nhiên 10% (MCAR) dữ liệu tại cột này.

In [3]:
# Define target and ground truth
target_col = 'age'
ground_truth = df_adult[target_col].copy()

# Simulate missing data (10%)
df_sim = df_adult.copy()
np.random.seed(42)
drop_ratio = 0.10
n_rows = len(df_sim)
mask_missing = np.random.rand(n_rows) < drop_ratio
df_sim.loc[mask_missing, target_col] = np.nan

print("\tMISSING DATA SIMULATION")
print(f"Missing values introduced: {mask_missing.sum():,}")
print(f"Missing ratio            : {(mask_missing.sum() / n_rows) * 100:.2f}%")

missing_indices = df_sim[df_sim[target_col].isna()].index

	MISSING DATA SIMULATION
Missing values introduced: 3,292
Missing ratio            : 10.11%


### **3.3 Implementation & Evaluation**

Nhóm tiến hành cài đặt 5 chiến lược (kèm các biến thể của k-NN) và đo lường *RMSE (Root Mean Square Error)*. Chỉ số RMSE càng thấp chứng tỏ giá trị nội suy càng bám sát với dữ liệu thực tế (Ground Truth) đã bị xóa.

In [4]:
# Prepare numerical data
df_num_sim = df_sim[num_cols].copy()
true_values = ground_truth.loc[missing_indices]
def compute_rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

# Evaluate imputation strategies
results = {}

# Mean
imputer = SimpleImputer(strategy='mean')
df_imputed = pd.DataFrame(imputer.fit_transform(df_num_sim), columns=num_cols)
results['Mean'] = compute_rmse(true_values, df_imputed.loc[missing_indices, target_col])

# Median
imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df_num_sim), columns=num_cols)
results['Median'] = compute_rmse(true_values, df_imputed.loc[missing_indices, target_col])

# Mode
imputer = SimpleImputer(strategy='most_frequent')
df_imputed = pd.DataFrame(imputer.fit_transform(df_num_sim), columns=num_cols)
results['Mode'] = compute_rmse(true_values, df_imputed.loc[missing_indices, target_col])

# k-NN (k = 3, 5, 10)
for k in [3, 5, 10]:
    imputer = KNNImputer(n_neighbors=k)
    df_imputed = pd.DataFrame(imputer.fit_transform(df_num_sim), columns=num_cols)
    results[f'KNN (k={k})'] = compute_rmse(true_values, df_imputed.loc[missing_indices, target_col])

# MICE
imputer = IterativeImputer(max_iter=10, random_state=42)
df_imputed = pd.DataFrame(imputer.fit_transform(df_num_sim), columns=num_cols)
results['MICE'] = compute_rmse(true_values, df_imputed.loc[missing_indices, target_col])

df_results = pd.DataFrame(list(results.items()), columns=['Strategy', 'RMSE'])
df_results = df_results.sort_values(by='RMSE').reset_index(drop=True)

print("\tIMPUTATION PERFORMANCE COMPARISON (RMSE)")
display(df_results.style.format({"RMSE": "{:.4f}"}).background_gradient(cmap='RdYlGn_r', subset=['RMSE']))

	IMPUTATION PERFORMANCE COMPARISON (RMSE)


,Strategy,RMSE
0,MICE,13.6168
1,Mean,13.7441
2,Median,13.8629
3,KNN (k=10),13.9149
4,Mode,14.0301
5,KNN (k=5),14.1506
6,KNN (k=3),14.4296


### **3.4 Performance Comparison & Strategy Selection**

Dựa trên cơ sở lý thuyết và kết quả thực nghiệm RMSE với 10.11% dữ liệu khuyết nhân tạo, nhóm thiết lập bảng đối chiếu chiến lược:

| Chiến lược | Giả định | Bảo toàn phân phối | Bảo toàn tương quan | Chi phí tính toán |
| :--- | :--- | :--- | :--- | :--- |
| **Mean** | MCAR, Normal | Rất thấp (Tạo chóp nhọn) | Kém | Cực thấp |
| **Median** | MCAR, Skewed | Thấp | Kém | Cực thấp |
| **Mode** | MCAR | Thấp | Kém | Cực thấp |
| **k-NN** | MAR / MCAR | Cao (dùng mẫu cục bộ) | Trung bình - Cao | Cao |
| **MICE** | MAR | Rất Cao | Cao (hồi quy đa biến)| Khá Cao |

**The Best Strategy: Multiple Imputation by Chained Equations (MICE)**

**Lý giải:**
Đối với các ứng dụng khoa học dữ liệu chuyên nghiệp, MICE được lựa chọn là phương pháp luận ưu việt nhất dựa trên 3 nguyên nhân:

1. **Empirical Evidence:** Kết quả RMSE chứng minh MICE là thuật toán nội suy chính xác nhất (RMSE = 13.61). Đáng chú ý, các phương pháp gán hằng số toàn cục (Mean, Median) lại vượt trội hơn hẳn k-NN. Về mặt toán học, hàm mục tiêu RMSE luôn "ưu ái" các giá trị trung tâm (như Mean) để tối thiểu hóa bình phương sai số của một phân phối tập trung như tuổi tác. Trong khi đó, thuật toán k-NN bị nhiễu do khoảng cách Euclidean bị chi phối nặng nề bởi các biến có thang đo lớn (Curse of Dimensionality) nếu chưa qua chuẩn hóa. MICE đã vượt qua được ảnh hưởng của Mean nhờ khả năng khai thác triệt để mối quan hệ tuyến tính/phi tuyến giữa biến mục tiêu và các biến quan sát khác.
2. **Handling of Uncertainty:** Khác với Mean/Median coi giá trị nội suy là sự thật tuyệt đối, MICE lồng ghép phương sai giữa các lần điền khuyết (between-imputation variance). Điều này ngăn chặn hiện tượng mô hình bị overconfident và bảo vệ khoảng tin cậy của dữ liệu.
3. **Distributional Fidelity:** Thông qua chuỗi phương trình hồi quy, MICE tái tạo lại phân phối mà không làm biến dạng (không tạo chóp nhọn nhân tạo như Mean/Median). Điều này cực kỳ quan trọng đối với các thuật toán nhạy cảm với phân phối ở các bước sau.

**Next Step:** Trong quy trình pipeline chính thức, nhóm sẽ sử dụng thuật toán *MICE (IterativeImputer)* để xử lý triệt để các giá trị thực sự bị khuyết nguyên bản trong bộ dữ liệu Adult Census (tại các thuộc tính phân loại như `workclass` và `occupation`, MICE sẽ được cấu hình sử dụng bộ phân loại Logistic/Random Forest Classifier thay vì Regressor).

## **4. Outlier Detection and Treatment**

**Mục tiêu:** Xây dựng cơ chế ensemble để xác định các quan sát bất thường trong các biến liên tục. Đánh giá sự chồng chéo giữa các thuật toán và đo lường tác động của việc loại bỏ ngoại lai đến hình dáng phân phối bằng kiểm định thống kê.

### **4.1 Theoretical Foundation & Mathematical Formulas**

#### 1. Interquartile Range (IQR) and Z-Score

* **IQR (Interquartile Range):** Là phương pháp phi tham số, tập trung vào 50% dữ liệu ở giữa, giúp nó cực kỳ bền vững trước các giá trị ngoại lai.
  * *Công thức:* $IQR = Q_3 - Q_1$.
  * *Ngưỡng cảnh báo:* 
      * $\text{Lower} = Q_1 - 1.5 \times IQR$ 
      * $\text{Upper} = Q_3 + 1.5 \times IQR$.
* **Z-Score:** Phương pháp tham số giả định dữ liệu có phân phối chuẩn. Định lượng khoảng cách từ một điểm đến trung bình mẫu tính bằng độ lệch chuẩn. Rất dễ bị hiện tượng "che khuất" (masking) do ngoại lai làm phình to phương sai.
  * *Công thức:* $Z = \frac{x - \mu}{\sigma}$. Các quan sát có $|Z| > 3$ được xem là ngoại lai.
  * *Độ chồng chéo:* IQR và Z-score thường có độ tương đồng Jaccard rất cao ($>0.8$) trong các phân phối chuẩn, nhưng Z-score dễ thất bại ở dữ liệu đa chiều.

#### 2. Isolation Forest (iForest)

* **Lý thuyết:** Thuật toán Ensemble cô lập các điểm dị thường bằng cách phân chia ngẫu nhiên không gian dữ liệu. Ngoại lai thường nằm ở các vùng thưa thớt, do đó chúng sẽ bị cô lập nhanh chóng với số rẽ nhánh ít hơn, tạo ra chiều dài đường đi ngắn hơn trong cây.
* **Công thức (Điểm dị thường):**
  $$s(x, n) = 2^{-\frac{E(h(x))}{c(n)}}$$
  *(Trong đó $E(h(x))$ là chiều dài đường đi trung bình, và $c(n)$ là chiều dài kỳ vọng của cây tìm kiếm nhị phân).* Điểm $s \approx 1$ ám chỉ ngoại lai.
* **Tác động của `contamination` (Tỉ lệ nhiễu):**
  * `0.01`: Cực kỳ thận trọng, chỉ bắt các điểm dị biệt rõ rệt (độ chính xác cao, Recall thấp).
  * `0.05`: Trạng thái tiêu chuẩn, cân bằng giữa rủi ro bỏ sót và báo động giả.
  * `0.10`: Aggressive, bắt các ngoại lai vi tế nhưng dễ gán nhãn sai vùng biên cụm.

#### 3. Local Outlier Factor (LOF)

* **Lý thuyết:** Phương pháp không giám sát dựa trên mật độ. Nó đánh giá mức độ cô lập của một điểm so với các "hàng xóm" lân cận. Rất mạnh trong việc tìm ngoại lai cục bộ ở các tập dữ liệu có mật độ phân tán không đồng đều.
* **Công thức toán học:**
  * *Khoảng cách khả đạt:* $\text{reach-dist}_k(p, o) = \max\{d_k(o), \text{dist}(p, o)\}$
  * *Mật độ khả đạt cục bộ (LRD):* $lrd_k(p) = \frac{|N_k(p)|}{\sum_{o \in N_k(p)} \text{reach-dist}_k(p, o)}$
  * *Chỉ số LOF:* $LOF_k(p) = \frac{\sum_{o \in N_k(p)} \frac{lrd_k(o)}{lrd_k(p)}}{|N_k(p)|}$. (Điểm số $\gg 1$ cảnh báo ngoại lai).
* **Tác động của `n_neighbors` ($k$):**
  * `10`: Nhạy cảm, tập trung vào vi cấu trúc cục bộ.
  * `20`: Mức "sweet spot", cân bằng và ổn định (độ chính xác thường $\sim 75-84\%$).
  * `50`: Tầm nhìn toàn cục (Global), làm mờ chi tiết, dễ mất đi năng lực phát hiện ngoại lai cục bộ.

#### 4. DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

* **Lý thuyết:** Nhận diện các vùng không gian mật độ cao thông qua bán kính $\epsilon$ và số điểm tối thiểu $MinPts$. Các điểm không thuộc bất kỳ cụm nào được gán nhãn là nhiễu (Noise / Outlier).
* **Hiện tượng "Outlier Cluster Cluster":** Nếu các điểm dị thường nằm gần nhau thỏa mãn $\epsilon$ và $MinPts$, chúng sẽ tự tạo thành một cụm riêng biệt thay vì bị coi là nhiễu, làm giảm khả năng nhận diện. Cần tinh chỉnh tham số cẩn thận.

#### 5. Kolmogorov-Smirnov (KS) Test

* **Lý thuyết:** Kiểm định phi tham số so sánh hàm phân phối tích lũy thực nghiệm (CDFs) trước và sau khi xử lý ngoại lai để đánh giá xem sự can thiệp có làm bóp méo bản chất dữ liệu hay không.
* **Công thức thống kê $D$:**
  $$D_{n,m} = \sup_x |F_1(x) - F_2(x)|$$
* **Biện luận:** Việc xóa bỏ thường trả về P-value $< 0.05$ (làm biến dạng phân phối). Trong khi cắt tỉa (Winsorization/Capping) thường bảo toàn phân phối tốt hơn (P-value $> 0.05$).

### **4.2 Algorithm Implementations & Detection Rates**

Nhóm tiến hành cài đặt 4 chiến lược phát hiện ngoại lai trên nhóm biến định lượng. Để đảm bảo các thuật toán dựa trên mật độ và khoảng cách (LOF, DBSCAN) hoạt động chính xác và không bị bóp méo bởi sự chênh lệch thang đo (ví dụ: `age` vs `capital.gain`), dữ liệu sẽ được chuẩn hóa tạm thời (`StandardScaler`) trước khi đưa vào thuật toán.

*(Lưu ý kỹ thuật: Để tránh hiện tượng tràn RAM (Out of Memory) trên tập dữ liệu $>30.000$ dòng, nhóm cấu hình thuật toán sử dụng cấu trúc `kd_tree` để tối ưu hóa bộ nhớ thay vì tính toán ma trận khoảng cách đối xứng toàn cục).*

#### 2. Nút thắt cổ chai: LOF và DBSCAN (High RAM)
Cả LOF và DBSCAN đều phải dò tìm lân cận (Nearest Neighbor Search). Phương pháp duyệt cạn (Brute-force) yêu cầu lưu trữ ma trận khoảng cách $N \times N$, đẩy độ phức tạp không gian lên $O(n^2)$. Với bộ dữ liệu Adult ($N \approx 32.000$), ma trận này sẽ đánh sập RAM hệ thống.

**Chiến lược Tối ưu hóa (Optimization Strategy):**
Để vượt qua rào cản phần cứng nhưng vẫn tuân thủ yêu cầu cài đặt thuật toán, nhóm áp dụng các giải pháp kỹ thuật sâu:
1. **Ép kiểu dữ liệu (Data Downcasting):** Chuyển toàn bộ mảng dữ liệu từ `float64` mặc định xuống `float32`, lập tức giảm 50% dung lượng RAM tiêu thụ.
2. **Spatial Indexing (Lập chỉ mục không gian):** Bắt buộc sử dụng cấu trúc `algorithm='kd_tree'` thay vì tính ma trận toàn cục. Cấu trúc cây giúp giảm độ phức tạp truy vấn xuống $O(n \log n)$.
3. **Tối ưu Kích thước lá (Leaf Size):** Cấu hình `leaf_size=50` (thay vì mặc định 30). Kích thước lá lớn hơn giúp giảm số lượng node trên cây, tiết kiệm dung lượng lưu trữ kiến trúc cây trong RAM.
4. **Vô hiệu hóa Đa luồng (Disable Multiprocessing):** Đặt `n_jobs=1` cho LOF và DBSCAN. Đa luồng (`n_jobs=-1`) trong Scikit-learn sẽ sinh (fork) ra nhiều tiến trình con, mỗi tiến trình sao chép một bản ma trận khoảng cách, là nguyên nhân chính gây tràn RAM đột ngột.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import scipy.stats as stats
import numpy as np
import pandas as pd

# Sử dụng danh sách biến số học từ các bước trước
outlier_features = ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss', 'hours.per.week']

# --- CHIẾN LƯỢC TỐI ƯU 1: DOWNCASTING ---
# Loại bỏ NaN và ép kiểu về float32 để giảm 50% RAM
X_outlier = df_adult[outlier_features].dropna().astype(np.float32)

# Chuẩn hóa Z-score tạm thời, kết quả cũng ép về float32
X_scaled = StandardScaler().fit_transform(X_outlier).astype(np.float32)

outlier_dict = {} # Dictionary lưu trữ index (vị trí) của các điểm ngoại lai

# --- 1. Z-Score (|Z| > 3) ---
z_scores = np.abs(stats.zscore(X_outlier))
outlier_dict['Z-Score'] = set(np.where((z_scores > 3).any(axis=1))[0])

# --- 2. IQR (Interquartile Range) ---
Q1 = X_outlier.quantile(0.25)
Q3 = X_outlier.quantile(0.75)
IQR = Q3 - Q1
iqr_condition = ((X_outlier < (Q1 - 1.5 * IQR)) | (X_outlier > (Q3 + 1.5 * IQR))).any(axis=1)
outlier_dict['IQR'] = set(np.where(iqr_condition)[0])

# --- 3. Isolation Forest (Thử nghiệm 3 mức contamination) ---
# iForest tiết kiệm RAM nên có thể dùng n_jobs=-1
for cont in [0.01, 0.05, 0.1]:
    iso = IsolationForest(contamination=cont, random_state=42, n_jobs=-1)
    preds = iso.fit_predict(X_outlier)
    outlier_dict[f'iForest_{cont}'] = set(np.where(preds == -1)[0])

# --- 4. LOF (Thử nghiệm 3 mức n_neighbors) ---
# CHIẾN LƯỢC TỐI ƯU 2 & 3 & 4: kd_tree, leaf_size=50, n_jobs=1 (tránh nhân bản RAM)
for n in [10, 20, 50]:
    lof = LocalOutlierFactor(
        n_neighbors=n, 
        contamination=0.05, 
        algorithm='kd_tree', 
        leaf_size=50, 
        n_jobs=1 
    )
    preds = lof.fit_predict(X_scaled)
    outlier_dict[f'LOF_{n}'] = set(np.where(preds == -1)[0])

# --- 5. DBSCAN (Noise points = -1) ---
# Tương tự LOF, sử dụng kd_tree và n_jobs=1. 
# eps=2.5 để tránh việc 1 điểm có quá nhiều hàng xóm gây tràn bộ nhớ đệm
dbscan = DBSCAN(
    eps=2.5, 
    min_samples=10, 
    algorithm='kd_tree', 
    leaf_size=50, 
    n_jobs=1
) 
preds = dbscan.fit_predict(X_scaled)
outlier_dict['DBSCAN'] = set(np.where(preds == -1)[0])

# --- Báo cáo tỉ lệ phát hiện ---
print(f"TỔNG SỐ QUAN SÁT KIỂM TRA: {len(X_outlier):,}\n")
print(f"{'Thuật toán':<15} | {'Số lượng ngoại lai':<18} | {'Tỉ lệ (%)':<10}")
print("-" * 50)
for name, idx_set in outlier_dict.items():
    rate = len(idx_set) / len(X_outlier) * 100
    print(f"{name:<15} | {len(idx_set):<18,} | {rate:.2f}%")

### **4.3. Overlap Analysis**

Để tránh việc "tin tưởng mù quáng" vào một thuật toán duy nhất có thể gây ra hiện tượng xóa nhầm dữ liệu bình thường (False Positives), nhóm tiến hành so sánh chéo các tập ngoại lai vừa tìm được bằng chỉ số **Jaccard Similarity** (Kích thước phần giao chia cho kích thước phần hợp). Mức độ chồng chéo cao chứng tỏ sự đồng thuận mạnh mẽ giữa các phương pháp tiếp cận khác nhau.

In [ ]:
def jaccard_similarity(set1, set2):
    if len(set1) == 0 and len(set2) == 0: return 1.0
    return len(set1.intersection(set2)) / len(set1.union(set2))

# Chọn các thuật toán tiêu biểu để tính Jaccard
methods = ['IQR', 'Z-Score', 'iForest_0.05', 'LOF_20', 'DBSCAN']
n_methods = len(methods)
jaccard_matrix = np.zeros((n_methods, n_methods))

for i, m1 in enumerate(methods):
    for j, m2 in enumerate(methods):
        jaccard_matrix[i, j] = jaccard_similarity(outlier_dict[m1], outlier_dict[m2])

# Trực quan hóa Heatmap Jaccard
plt.figure(figsize=(8, 6))
sns.heatmap(jaccard_matrix, annot=True, xticklabels=methods, yticklabels=methods, 
            cmap="YlGnBu", fmt=".2f", vmin=0, vmax=1)
plt.title("Jaccard Similarity Giữa Các Thuật Toán Ngoại Lai", fontweight='bold', pad=15)
plt.show()

# Xây dựng cơ chế đồng thuận (Consensus Ensemble)
# Giữ lại các index bị đánh dấu là ngoại lai bởi ÍT NHẤT 3 phương pháp (Major Voting)
all_outliers = []
for m in methods:
    all_outliers.extend(list(outlier_dict[m]))
    
from collections import Counter
outlier_counts = Counter(all_outliers)
consensus_outliers = [idx for idx, count in outlier_counts.items() if count >= 3]

print(f"Số lượng ngoại lai đồng thuận (Consensus Outliers): {len(consensus_outliers):,} "
      f"({len(consensus_outliers)/len(X_outlier)*100:.2f}%)")

### **4.4. Distributional Impact via KS Test**

Nếu loại bỏ (Drop) trực tiếp tập ngoại lai đồng thuận này, liệu chúng ta có làm thay đổi hoàn toàn bản chất phân phối của dữ liệu gốc hay không? Nhóm sử dụng **Two-Sample Kolmogorov-Smirnov Test** để so sánh phân phối của cột `age` (đại diện) trước và sau khi xóa.

* $H_0$: Phân phối trước và sau khi xóa ngoại lai là giống nhau.
* $H_1$: Xóa ngoại lai làm thay đổi vĩnh viễn phân phối dữ liệu (P-value $< 0.05$).

In [ ]:
# Tập dữ liệu trước khi xóa
feature_to_test = 'age'
dist_before = X_outlier[feature_to_test].values

# Tập dữ liệu sau khi xóa ngoại lai đồng thuận
dist_after = X_outlier.drop(index=X_outlier.index[consensus_outliers])[feature_to_test].values

# Chạy kiểm định KS
ks_stat, p_value = stats.ks_2samp(dist_before, dist_after)

print("=== KẾT QUẢ KIỂM ĐỊNH KOLMOGOROV-SMIRNOV ===")
print(f"Đặc trưng kiểm định: {feature_to_test}")
print(f"Kích thước mẫu trước: {len(dist_before):,} | Kích thước mẫu sau: {len(dist_after):,}")
print(f"KS Statistic (Khoảng cách D lớn nhất): {ks_stat:.4f}")
print(f"P-value: {p_value:.4e}")

if p_value < 0.05:
    print("\nKết luận: P-value < 0.05. Bác bỏ H0.")
    print("=> CẢNH BÁO: Việc xóa ngoại lai làm BIẾN DẠNG đáng kể phân phối gốc.")
else:
    print("\nKết luận: P-value >= 0.05. Chấp nhận H0.")
    print("=> AN TOÀN: Việc can thiệp không làm thay đổi bản chất phân phối.")

### **4.5. Distributional Impact via KS Test**

Từ ma trận Jaccard và kiểm định KS, nhóm rút ra nhận định chuyên môn:
1. **Sự mâu thuẫn về bản chất:** Các thuật toán định chuẩn toàn cục (Z-Score, IQR) có sự đồng thuận với nhau, nhưng rất lệch pha với các thuật toán dựa trên cấu trúc hình học không gian (LOF). Điều này chứng minh rằng "Ngoại lai trong thống kê chưa chắc đã là ngoại lai trong cấu trúc phân bố không gian".
2. **Hệ quả của KS Test:** Việc xóa bỏ (Deletion) nhóm ngoại lai đồng thuận đã bóp méo phân phối gốc của biến số học (P-value $\approx 0$). 
3. **Quyết định (Treatment Strategy):** Trong bài toán phân loại thu nhập, tầng lớp siêu giàu (`capital.gain` cực lớn) tuy là ngoại lai về mặt toán học nhưng lại là tín hiệu quan trọng nhất để dự đoán nhãn `>50K`. Do đó, nhóm quyết định **KHÔNG XÓA** tập ngoại lai này. Thay vào đó, ở Mục 3, nhóm sẽ sử dụng thuật toán **Robust Scaling** hoặc **Quantile Transformer** để kiềm chế biên độ của chúng mà không làm mất mát lượng thông tin phân loại quý giá này.

## **5. Distribution Analysis**

### **5.1. Overview of D'Agostino-Pearson Omnibus Test**

### **5.2. Hypothesis Formulation**

Tương tự các kiểm định tính chuẩn khác, giả thuyết được đặt ra là:
* $H_0$ (Giả thuyết không): Mẫu dữ liệu được rút ra từ một tổng thể có phân phối chuẩn.
* $H_1$ (Giả thuyết thay thế hay đối thuyết): Mẫu dữ liệu không tuân theo phân phối chuẩn.

### **5.3 Core Mathematical Algorithms**

### **5.5 Implementation & Distribution Classification**

### **5.6 Scaling Strategy Proposal**

#### Distributional Characteristics

#### Implications for Feature Scaling

#### Recommended Scaling Strategy

## **6. Multivariate Correlation & Multicollinearity Analysis**

### **6.1 Theoretical Foundation: Correlation Metrics**

### **6.2 Bivariate Analysis & Heatmap Visualization**

### **6.3 Multicollinearity Detection & Critical Analysis**

### **6.4 Proposed Treatments for Multicollinearity**

---